# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
This dataset's source is provided via a Croissant schema URL (see below). We will:
- Load and inspect the dataset using its Croissant metadata.
- Review the available record sets and field structures (all using their `@id`).
- Extract and process data using record set and field `@id`s.
- Perform simple exploratory data analysis and visualizations.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# The metadata object contains the Croissant dataset metadata
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as discovered in the Croissant metadata.

We'll enumerate all record sets, then for one record set, inspect its fields and field `@id`s for reference.

In [ ]:
# List all available record sets in the dataset via their @id
record_sets = []
print("Available record sets and their @id:")
for rs in metadata.record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    record_sets.append(rs['@id'])

# For demonstration, pick the first record set for field exploration
if record_sets:
    example_record_set_id = record_sets[0]
    rs_obj = [r for r in metadata.record_sets if r['@id'] == example_record_set_id][0]
    print(f"\nFields for record set {example_record_set_id}:")
    for field in rs_obj['fields']:
        print(f"  - {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

**All references are made using `@id` fields for record sets and fields.**

Let's load all available record sets.

In [ ]:
# If record sets were found, we'll extract all into pandas DataFrames.
dataframes = {}

for rsid in record_sets:
    print(f"Loading record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print("  No records available or record set is empty.")

# Preview the first DataFrame (if any) and its columns
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows from record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we'll pick a numeric field by its `@id`, filter and normalize it, and optionally group by a categorical field if present.

In [ ]:
### User should update the following field @id according to available record set and numeric fields listed above ###
rs_id = first_rs_id  # Example: Use the first record set loaded

df = dataframes[rs_id].copy() if rs_id in dataframes else None

# Inspect fields/columns to identify numeric fields for analysis
print(f"Fields in {rs_id}: {df.columns.tolist() if df is not None else 'No DataFrame'}")

# Set field IDs for demonstration; adjust these to actual field IDs available in dataset
if df is not None and not df.empty:
    # Try to pick a plausible numeric field (e.g., molecular_weight, coverage, peptide_count)
    candidates = [c for c in df.columns if df[c].dtype.kind in 'iufc']
    if not candidates:
        # Try to convert any columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        candidates = [c for c in df.columns if df[c].dtype.kind in 'iufc']
    if candidates:
        numeric_field = candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a non-numeric field, if present
        group_fields = [c for c in df.columns if df[c].dtype == 'object']
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped (mean) {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            group_field = None
    else:
        print("No numeric fields suitable for EDA found.")
else:
    numeric_field = None
    group_field = None
    print("No DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we show a histogram of a chosen numeric field, if one was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group_field is identified, show boxplot
    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all dataset entities by their Croissant `@id` as required. We performed basic data loading, inspection, simple filtering, normalization, grouping, and visualization. 

- All entities such as record sets and fields were referenced using their `@id`s for unambiguous reproducibility.
- You can now extend this notebook to perform your own statistical analysis or modeling on the dataset, using any of the available fields as referenced by their `@id`.

For further details, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and the dataset's [schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).